# Merge Hofstede IDV into Team Panels

Reads `data/gravity/hofstede.csv` and the raw match Excel to compute
`ic_team = (IDV_p1 + IDV_p2) / 2` for each team-observation.
Appends `ic_team` (and demeaned `ic_team_dm`) to `team_gs_panel.csv` and
`tiebreak_panel.csv`.

**Pipeline position**: runs after `final_ds.ipynb` and `tiebreak_panel.ipynb`.

In [ ]:
import os
import pandas as pd

ROOT         = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
EXCEL_PATH   = os.path.join(ROOT, 'data', 'atp', 'men_matches_with_ranks_cleaned.xlsx')
HOFSTEDE_PATH= os.path.join(ROOT, 'data', 'gravity', 'hofstede.csv')
GS_PANEL     = os.path.join(ROOT, 'data', 'atp', 'team_gs_panel.csv')
TB_PANEL     = os.path.join(ROOT, 'data', 'atp', 'tiebreak_panel.csv')
print('Paths set.')


## 1. Build match-level IDV lookup from raw Excel

In [ ]:
# Load only the ISO3 columns we need from the raw Excel
iso_cols = ['match_id',
            'winners_p1_iso3','winners_p2_iso3',
            'losers_p1_iso3', 'losers_p2_iso3']
raw = pd.read_excel(EXCEL_PATH, sheet_name='players_list', usecols=iso_cols)
print(f'Raw rows: {len(raw)}')

# Load Hofstede IDV
hof = pd.read_csv(HOFSTEDE_PATH)[['iso3','idv']].dropna(subset=['idv'])
idv_map = hof.set_index('iso3')['idv'].to_dict()
print(f'IDV entries: {len(idv_map)}')

def lookup(iso):
    if pd.isna(iso): return float('nan')
    return idv_map.get(str(iso), float('nan'))

raw['idv_wp1'] = raw['winners_p1_iso3'].apply(lookup)
raw['idv_wp2'] = raw['winners_p2_iso3'].apply(lookup)
raw['idv_lp1'] = raw['losers_p1_iso3'].apply(lookup)
raw['idv_lp2'] = raw['losers_p2_iso3'].apply(lookup)

raw['ic_team_winners'] = (raw['idv_wp1'] + raw['idv_wp2']) / 2
raw['ic_team_losers']  = (raw['idv_lp1'] + raw['idv_lp2']) / 2

match_ic = raw[['match_id','ic_team_winners','ic_team_losers']].copy()
print(f'Missing ic_team_winners: {match_ic["ic_team_winners"].isna().sum()}')
print(f'Missing ic_team_losers:  {match_ic["ic_team_losers"].isna().sum()}')
match_ic.head(3)


## 2. Add ic_team to team_gs_panel.csv

In [ ]:
gs = pd.read_csv(GS_PANEL)
if 'ic_team' in gs.columns:
    gs = gs.drop(columns=['ic_team','ic_team_dm'], errors='ignore')

gs = gs.merge(match_ic, on='match_id', how='left')

# Assign ic_team based on whether this row is the winner (win==1) or loser
gs['ic_team'] = gs.apply(
    lambda r: r['ic_team_winners'] if r['win'] == 1 else r['ic_team_losers'], axis=1)
gs = gs.drop(columns=['ic_team_winners','ic_team_losers'])

# Demean ic_team across the full sample (GS panel, for consistency with regressions)
ic_mean = gs['ic_team'].mean()
gs['ic_team_dm'] = gs['ic_team'] - ic_mean
print(f'ic_team mean (used for demeaning): {ic_mean:.2f}')
print(f'Missing ic_team in GS panel: {gs["ic_team"].isna().sum()}')

gs.to_csv(GS_PANEL, index=False)
print(f'Saved team_gs_panel.csv  ({len(gs)} rows)')


## 3. Add ic_team to tiebreak_panel.csv

In [ ]:
tb = pd.read_csv(TB_PANEL)
if 'ic_team' in tb.columns:
    tb = tb.drop(columns=['ic_team','ic_team_dm'], errors='ignore')

tb = tb.merge(match_ic, on='match_id', how='left')
tb['ic_team'] = tb.apply(
    lambda r: r['ic_team_winners'] if r['win'] == 1 else r['ic_team_losers'], axis=1)
tb = tb.drop(columns=['ic_team_winners','ic_team_losers'])

# Use the same mean as the GS panel for demeaning
tb['ic_team_dm'] = tb['ic_team'] - ic_mean

print(f'Missing ic_team in tiebreak panel: {tb["ic_team"].isna().sum()}')
tb.to_csv(TB_PANEL, index=False)
print(f'Saved tiebreak_panel.csv ({len(tb)} rows)')


## 4. Coverage report

In [ ]:
print('=== GS Panel ===')
print(gs[['ic_team','ic_team_dm']].describe().round(2))
print()
print('=== Tiebreak Panel ===')
print(tb[['ic_team','ic_team_dm']].describe().round(2))
